# Modelo-07 — Construção da Tabela Mestra do Sistema PSD

## Objetivo

Este notebook tem como objetivo integrar e padronizar os dados históricos do sistema de Portas de Plataforma (PSD), preparando uma tabela mestra para as etapas posteriores de processamento e treinamento do `CatBoostClassifier`.

A unidade de análise será o sistema PSD de cada **estação e plataforma** em uma determinada data. Dessa forma, os alarmes e as manutenções dos diferentes componentes — como PDMs, MDUs, sensores, controladores e portas de emergência — poderão representar conjuntamente o estado operacional daquele sistema.

Neste notebook serão realizadas somente a leitura, integração, padronização e auditoria das fontes. A criação das features e do alvo — ocorrência de uma falha nos 7 dias seguintes — será feita no notebook de processamento, evitando misturar a preparação dos eventos com a modelagem.

## Hipóteses que orientam a construção da tabela

A tabela mestra deve preservar os campos necessários para testar e transformar as evidências levantadas na Sprint 1 em features temporais. A inclusão de uma informação na tabela não significa que ela será obrigatoriamente utilizada pelo modelo.

| Hipótese | Resultado da investigação | Informação que deve ser preservada |
|---|---|---|
| H1 — padrões por horário | Confirmada | Horário do alarme, duração e identificação do PSD |
| H2 — regime semanal | Confirmada | Data do evento para derivar dia da semana e domingo |
| H3 — fluxo de passageiros | Refutada | Dados de catraca não serão integrados na primeira versão |
| H4 — aumento de alarmes antes da corretiva | Não confirmada | Tipo e horário dos alarmes serão preservados, sem assumir que a contagem sempre cresce antes da falha |
| H5 — tempo de normalização | Possui sinal, mas é insuficiente isoladamente | Início, normalização, duração e estação |
| H6 — posição central ou extremidade | Efeito prático pequeno | A posição da porta não será uma feature prioritária no modelo agregado por plataforma |
| H7 — predominância de falhas mecânicas | Sustentada | Causa, defeito, objeto e ação das manutenções históricas |

> **Prevenção de vazamento temporal:** causa, defeito e ação só poderão gerar features depois que a respectiva informação estiver disponível no histórico. Esses campos nunca poderão descrever a própria falha que o modelo está tentando prever.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Semente: {RANDOM_SEED}")

Pandas: 3.0.5
NumPy: 2.5.1
Semente: 42


In [2]:
df_alarmes = pd.read_csv("../../data/Alarmes/alarmes_trata_limpos_consolidado.csv")
df_corretiva_emergencial = pd.read_excel("../../data/Corretiva Emergencial/processados/ZM01_limpa.xlsx")
df_corretiva_programada = pd.read_excel("../../data/Corretiva Programada/processados/ZM02_limpa.xlsx")
df_preventiva = pd.read_csv("../../data/Preventiva/preventiva_zm03_limpa.csv")

print("Alarmes:", df_alarmes.shape)
print("Corretivas emergenciais:", df_corretiva_emergencial.shape)
print("Corretivas programadas:", df_corretiva_programada.shape)
print("Preventivas:", df_preventiva.shape)

Alarmes: (742506, 15)
Corretivas emergenciais: (10543, 27)
Corretivas programadas: (1163, 27)
Preventivas: (569, 19)


In [3]:
bases = {
    "alarmes": df_alarmes,
    "corretiva_emergencial": df_corretiva_emergencial,
    "corretiva_programada": df_corretiva_programada,
    "preventiva": df_preventiva,
}

for nome, df in bases.items():
    print(f"\n{nome.upper()} — {df.shape[0]:,} linhas × {df.shape[1]} colunas")

    resumo_colunas = pd.DataFrame({
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": df.isna().sum().values,
        "nulos_%": (df.isna().mean().values * 100).round(2),
    })

    display(resumo_colunas)
    display(df.head(3))


ALARMES — 742,506 linhas × 15 colunas


,coluna,tipo,nulos,nulos_%
0,TAG,str,0,0.00
1,Horário do Alarme,str,0,0.00
2,Estação,str,0,0.00
3,Porta,float64,170,0.02
4,Alarme,str,0,0.00
5,Horário da Normalizacao,str,0,0.00
6,Tempo de Normalização,str,0,0.00
7,tempo_normalizacao_seg,int64,0,0.00
8,dt_alarme,str,0,0.00
9,dt_normalizacao,str,0,0.00


,TAG,Horário do Alarme,Estação,Porta,Alarme,Horário da Normalizacao,Tempo de Normalização,tempo_normalizacao_seg,dt_alarme,dt_normalizacao,duracao_calculada_seg,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido
0,PSD_FAL_FAL2_PDM09_OBS,2026-05-12 08:16:52.657,FAL,9.00,OBS,2026-05-12 08:16:53.670,"0 dias, 00:00:01",1,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670,1.01,FAL,FAL2,9.00,OBS
1,PSD_LUZ_LUZ1_PDM08_OBS,2026-05-12 08:15:09.913,LUZ,8.00,OBS,2026-05-12 08:15:10.917,"0 dias, 00:00:01",1,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917,1.00,LUZ,LUZ1,8.00,OBS
2,PSD_FAL_FAL1_PDM15_OBS,2026-05-12 08:14:06.657,FAL,15.00,OBS,2026-05-12 08:14:07.670,"0 dias, 00:00:01",1,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670,1.01,FAL,FAL1,15.00,OBS



CORRETIVA_EMERGENCIAL — 10,543 linhas × 27 colunas


,coluna,tipo,nulos,nulos_%
0,linha_origem,int64,0,0.00
1,tipo_de_nota,str,0,0.00
2,nota,int64,0,0.00
3,ordem,float64,1,0.01
4,perfil_do_catalogo,str,0,0.00
5,descricao,str,0,0.00
6,local_de_instalacao,str,0,0.00
7,centro_de_trabalho_responsavel,str,0,0.00
8,inicio_avaria_datetime,datetime64[us],0,0.00
9,fim_avaria_datetime,datetime64[us],326,3.09


...

,coluna,tipo,nulos,nulos_%
0,linha_origem,int64,0,0.00
1,tipo_de_nota,str,0,0.00
2,nota,int64,0,0.00
3,ordem,float64,5,0.43
4,perfil_do_catalogo,str,0,0.00
5,descricao,str,0,0.00
6,local_de_instalacao,str,0,0.00
7,centro_de_trabalho_responsavel,str,0,0.00
8,inicio_avaria_datetime,datetime64[us],0,0.00
9,fim_avaria_datetime,datetime64[us],39,3.35


,linha_origem,tipo_de_nota,nota,ordem,perfil_do_catalogo,descricao,local_de_instalacao,centro_de_trabalho_responsavel,inicio_avaria_datetime,fim_avaria_datetime,duracao_avaria_horas,encerramento_datetime,indisponivel_flag,analisada_flag,ordem_ausente_flag,ordem_repetida_flag,nota_repetida_flag,duracao_extrema_flag,duracao_zero_flag,status_do_usuario_da_ordem,texto_longo_da_ordem,status_do_sistema_da_ordem,sintoma,objeto,defeito,causa,acao
0,2,Z2,15069252,"54,081,268.00",6PSD,Porta PCC-PSD a,SP04-6-PSD-BUT,SP04IBUT,2026-01-25 08:20:00,2026-01-25 09:30:00,1.17,2026-01-25 16:50:18,False,False,False,False,False,False,False,LIBO EXDO,Porta PCC-PSD 25.01.2026 08:29:08 BRAZIL (28-01325) 25.01.2026 16:43:35 BRAZIL (28-04512) Realizado a instalação de ...,ENTE CNPA CAPC NOLQ SCDM,PDM NÃO ABRE,"ESTRUTURA, ACABAMENTO, E ACESSÓRIOS DO P",PORTA DESAJUSTADA,FALTA FIXACAO,FIXADO
1,3,Z2,14647619,"53,659,484.00",6PSD,PDMs plat 1 e 2 com falha momentânea.,SP04-6-PSD-BUT-P1S1,SP04IBUT,2025-05-30 09:30:00,2025-06-01 04:00:00,42.50,2025-06-01 04:36:44,False,False,False,False,False,False,False,LIBO EXDO,PDMs plat 1 e 2 com falha momentânea. 30.05.2025 14:24:19 BRAZIL DENIS RODRIGUES DO NASCIMENTO (28-04212) O CCO info...,ENTE CNPA CAPC NOLQ SCDM,MAU CONTATO,SWITCK DE COMUNICAÇÃO,SEM CONEXAO ELETRICA,ERRO DE SOFTWARE,RESETADO E TESTADO
2,4,Z2,14682623,"53,694,340.00",6PSD,Rele painel SSE intermitente,SP04-6-PSD-BUT-P2S1-PSSE2,SP04IBUT,2025-06-19 02:00:00,2025-06-19 04:30:00,2.50,2025-06-19 05:10:12,False,False,False,False,False,False,False,LIBO EXDO,Rele painel SSE intermitente 19.06.2025 05:05:07 BRAZIL D. DE MELO SOARES (28-00570) Tel. 55 11 458.. Realizado a su...,ENTE CNPA CAPC NOLQ SCDM,FACHADA PSD SEM ALIMENTAÇÃO,DISPOSITIVO DE PROTECAO,CONVERSOR DE TENSAO AVARIADO,OPERACAO INDEVIDA,SUBSTITUIDO(A)



PREVENTIVA — 569 linhas × 19 colunas


,coluna,tipo,nulos,nulos_%
0,Ordem,int64,0,0.00
1,Tipo de ordem,str,0,0.00
2,Centro de trabalho,str,0,0.00
3,Local de instalação,str,0,0.00
4,Texto breve,str,0,0.00
5,Plano de manutenção,int64,0,0.00
6,Status usuário,str,0,0.00
7,Status do sistema,str,0,0.00
8,Data-base do início,str,0,0.00
9,Hr.início programada,str,0,0.00


,Ordem,Tipo de ordem,Centro de trabalho,Local de instalação,Texto breve,Plano de manutenção,Status usuário,Status do sistema,Data-base do início,Hr.início programada,Data-base do fim,Fim-base (hora),DT_Inicio,DT_Fim,duracao_h,Estacao_Centro,Estacao_Local,Estacao,cancelada
0,54164639,ZM03,SP04IPTA,SP04-6-PSD-PTA,PSD - RACK DO PCC (AR COND.) PTA P1,47029,LIBO EXDO ANEX,ENTE CNPA CAPC NOLQ SCDM,2026-03-23,21:30:00,2026-03-24,04:30:00,2026-03-23 21:30:00,2026-03-24 04:30:00,7.00,PTA,PTA,PTA,False
1,54164638,ZM03,SP04IFAL,SP04-6-PSD-FAL,PSD - RACK DO PCC (AR COND.) FAL P1,47028,LIBO EXDO,ENTE CNPA CAPC MatC NOLQ,2026-03-24,21:30:00,2026-03-25,04:30:00,2026-03-24 21:30:00,2026-03-25 04:30:00,7.00,FAL,FAL,FAL,False
2,54152224,ZM03,SP04IPIN,SP04-6-PSD-PIN,PSD - RACK DO PCC (AR COND.) PIN P1,47027,LIBO EXDO ANEX,ENTE CNPA CAPC NOLQ SCDM,2026-03-17,21:30:00,2026-03-18,04:30:00,2026-03-17 21:30:00,2026-03-18 04:30:00,7.00,PIN,PIN,PIN,False


In [4]:
import unicodedata

def padronizar_nome_coluna(nome):
    nome = unicodedata.normalize("NFKD", str(nome).strip().lower())
    nome = nome.encode("ascii", errors="ignore").decode("utf-8")
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    return nome.strip("_")

for nome_base, df in bases.items():
    colunas_originais = df.columns.tolist()
    colunas_padronizadas = [padronizar_nome_coluna(coluna) for coluna in colunas_originais]

    if len(colunas_padronizadas) != len(set(colunas_padronizadas)):
        raise ValueError(f"A padronização gerou colunas duplicadas na base {nome_base}.")

    df.columns = colunas_padronizadas
    print(f"{nome_base}: {df.columns.tolist()}")

alarmes: ['tag', 'horario_do_alarme', 'estacao', 'porta', 'alarme', 'horario_da_normalizacao', 'tempo_de_normalizacao', 'tempo_normalizacao_seg', 'dt_alarme', 'dt_normalizacao', 'duracao_calculada_seg', 'estacao_extraida', 'plataforma_extraida', 'porta_extraida', 'tipo_evento_extraido']
corretiva_emergencial: ['linha_origem', 'tipo_de_nota', 'nota', 'ordem', 'perfil_do_catalogo', 'descricao', 'local_de_instalacao', 'centro_de_trabalho_responsavel', 'inicio_avaria_datetime', 'fim_avaria_datetime', 'duracao_avaria_horas', 'analise_datetime', 'encerramento_datetime', 'indisponivel_flag', 'falha_flag', 'analisada_flag', 'ordem_ausente_flag', 'ordem_repetida_flag', 'comentarios_classificacao_engenharia', 'status_do_usuario_da_ordem', 'texto_longo_da_ordem', 'status_do_sistema_da_ordem', 'sintoma', 'objeto', 'defeito', 'causa', 'acao']
corretiva_programada: ['linha_origem', 'tipo_de_nota', 'nota', 'ordem', 'perfil_do_catalogo', 'descricao', 'local_de_instalacao', 'centro_de_trabalho_responsa

In [5]:
campos_de_identificacao = {
    "alarmes": ["estacao_extraida", "plataforma_extraida", "porta_extraida"],
    "corretiva_emergencial": ["local_de_instalacao"],
    "corretiva_programada": ["local_de_instalacao"],
    "preventiva": ["estacao", "local_de_instalacao"],
}

auditoria_identificacao = []

for nome_base, campos in campos_de_identificacao.items():
    df = bases[nome_base]

    for campo in campos:
        valores = df[campo].dropna().astype(str).str.strip()
        exemplos = valores.value_counts().head(10).index.tolist()

        auditoria_identificacao.append({
            "base": nome_base,
            "campo": campo,
            "preenchidos": valores.shape[0],
            "nulos": int(df[campo].isna().sum()),
            "valores_unicos": int(valores.nunique()),
            "10_valores_mais_frequentes": exemplos,
        })

display(pd.DataFrame(auditoria_identificacao))

,base,campo,preenchidos,nulos,valores_unicos,10_valores_mais_frequentes
0,alarmes,estacao_extraida,742506,0,11,"[FRA, PIN, HIG, BUT, MBI, REP, LUZ, PTA, VSO, FAL]"
1,alarmes,plataforma_extraida,742506,0,22,"[FRA2, PIN2, FRA1, PIN1, BUT1, HIG2, MBI1, HIG1, BUT2, MBI2]"
2,alarmes,porta_extraida,742336,170,24,"[9.0, 10.0, 16.0, 7.0, 18.0, 15.0, 14.0, 12.0, 8.0, 17.0]"
3,corretiva_emergencial,local_de_instalacao,10543,0,691,"[SP04-6-PSD-REP-PLT1, SP04-6-PSD-PIN-PLT1, SP04-6-PSD-REP-PLT2, SP04-6-PSD-REP-PLT2-PP21PDM, SP04-6-PSD-REP-PLT2-PP0..."
4,corretiva_programada,local_de_instalacao,1163,0,582,"[SP04-6-PSD-PTA-SLBT-PCC, SP04-6-PSD-REP-PLT1, SP04-6-PSD-HIG-PLT1-PP17PDM, SP04-6-PSD-MBI-PLT2-PP24PDM, SP04-6-PSD-..."
5,preventiva,estacao,569,0,12,"[GERAL, FAL, PTA, PIN, BUT, MBI, VSO, FRA, REP, HIG]"
6,preventiva,local_de_instalacao,569,0,28,"[SP04-6-PSD, SP04-6-PSD-FAL, SP04-6-PSD-PIN, SP04-6-PSD-BUT, SP04-6-PSD-PTA, SP04-6-PSD-PVS, SP04-6-PSD-VSO-PLT1, SP..."


In [6]:
def extrair_plataforma_local(local):
    local = local.astype("string").str.upper().str.strip()
    plataforma = local.str.extract(r"(?:^|-)PLT0*([12])(?:-|$)", expand=False)
    plataforma_legada = local.str.extract(r"(?:^|-)P([12])S\d+(?:-|$)", expand=False)
    return pd.to_numeric(plataforma.fillna(plataforma_legada), errors="coerce").astype("Int8")

def extrair_componente(texto):
    return (
        texto.astype("string")
        .str.upper()
        .str.extract(r"(PDM|MDU|PSE|PCC|SSE)", expand=False)
    )

estacoes_validas = set(
    df_alarmes["estacao_extraida"].dropna().astype(str).str.upper().str.strip()
)

df_alarmes["estacao_padrao"] = df_alarmes["estacao_extraida"].astype("string").str.upper().str.strip()
df_alarmes["plataforma_padrao"] = pd.to_numeric(
    df_alarmes["plataforma_extraida"].astype("string").str.extract(r"([12])$", expand=False),
    errors="coerce",
).astype("Int8")
df_alarmes["porta_padrao"] = pd.to_numeric(df_alarmes["porta_extraida"], errors="coerce").astype("Int16")
df_alarmes["componente_padrao"] = extrair_componente(df_alarmes["tag"])

for df in [df_corretiva_emergencial, df_corretiva_programada]:
    local = df["local_de_instalacao"].astype("string").str.upper().str.strip()
    estacao_extraida = local.str.extract(r"(?:^|-)PSD-([A-Z]{3})(?:-|$)", expand=False)
    df["estacao_padrao"] = estacao_extraida.where(estacao_extraida.isin(estacoes_validas), pd.NA)
    df["plataforma_padrao"] = extrair_plataforma_local(local)
    df["porta_padrao"] = pd.to_numeric(
        local.str.extract(r"PP0*(\d{1,2})PDM", expand=False), errors="coerce"
    ).astype("Int16")
    df["componente_padrao"] = extrair_componente(local)

estacao_preventiva = df_preventiva["estacao"].astype("string").str.upper().str.strip()
df_preventiva["estacao_padrao"] = estacao_preventiva.where(estacao_preventiva.isin(estacoes_validas), pd.NA)
df_preventiva["plataforma_padrao"] = extrair_plataforma_local(df_preventiva["local_de_instalacao"])
df_preventiva["porta_padrao"] = pd.Series(pd.NA, index=df_preventiva.index, dtype="Int16")
df_preventiva["componente_padrao"] = extrair_componente(df_preventiva["local_de_instalacao"])

for nome_base, df in bases.items():
    cobertura = df[["estacao_padrao", "plataforma_padrao", "porta_padrao", "componente_padrao"]].notna().mean().mul(100).round(2)
    print(f"{nome_base}: {cobertura.to_dict()}")

alarmes: {'estacao_padrao': 100.0, 'plataforma_padrao': 100.0, 'porta_padrao': 99.98, 'componente_padrao': 99.98}
corretiva_emergencial: {'estacao_padrao': 99.8, 'plataforma_padrao': 97.91, 'porta_padrao': 87.86, 'componente_padrao': 89.92}
corretiva_programada: {'estacao_padrao': 99.31, 'plataforma_padrao': 94.41, 'porta_padrao': 66.64, 'componente_padrao': 90.71}
preventiva: {'estacao_padrao': 81.9, 'plataforma_padrao': 55.01, 'porta_padrao': 0.0, 'componente_padrao': 0.0}


In [7]:
colunas_temporais = {
    "alarmes": "dt_alarme",
    "corretiva_emergencial": "inicio_avaria_datetime",
    "corretiva_programada": "inicio_avaria_datetime",
    "preventiva": "dt_inicio",
}

auditoria_temporal = []

for nome_base, coluna_origem in colunas_temporais.items():
    df = bases[nome_base]
    df["data_hora_evento"] = pd.to_datetime(df[coluna_origem], errors="coerce")
    df["data_evento"] = df["data_hora_evento"].dt.normalize()

    auditoria_temporal.append({
        "base": nome_base,
        "coluna_origem": coluna_origem,
        "data_minima": df["data_hora_evento"].min(),
        "data_maxima": df["data_hora_evento"].max(),
        "datas_invalidas_ou_nulas": int(df["data_hora_evento"].isna().sum()),
    })

display(pd.DataFrame(auditoria_temporal))

,base,coluna_origem,data_minima,data_maxima,datas_invalidas_ou_nulas
0,alarmes,dt_alarme,2023-02-27 06:26:56.897,2026-05-12 08:16:52.657,0
1,corretiva_emergencial,inicio_avaria_datetime,2023-01-01 02:39:00.000,2026-04-01 22:43:00.000,0
2,corretiva_programada,inicio_avaria_datetime,2023-01-03 03:55:59.000,2026-03-28 14:15:00.000,0
3,preventiva,dt_inicio,2022-12-26 05:30:00.000,2026-03-30 21:30:00.000,0


In [8]:
DATA_INICIO_COMUM = max(
    df["data_evento"].min() for df in bases.values()
).normalize()

DATA_FIM_COMUM = min(
    df["data_evento"].max() for df in bases.values()
).normalize()

if DATA_INICIO_COMUM > DATA_FIM_COMUM:
    raise ValueError("As bases não possuem um período temporal em comum.")

auditoria_janela = []

for nome_base, df in bases.items():
    auditoria_janela.append({
        "base": nome_base,
        "registros_antes_da_janela": int((df["data_evento"] < DATA_INICIO_COMUM).sum()),
        "registros_dentro_da_janela": int(df["data_evento"].between(DATA_INICIO_COMUM, DATA_FIM_COMUM).sum()),
        "registros_depois_da_janela": int((df["data_evento"] > DATA_FIM_COMUM).sum()),
    })

print(f"Período comum: {DATA_INICIO_COMUM.date()} até {DATA_FIM_COMUM.date()}")
print(f"Total de dias: {(DATA_FIM_COMUM - DATA_INICIO_COMUM).days + 1:,}")
display(pd.DataFrame(auditoria_janela))

Período comum: 2023-02-27 até 2026-03-28
Total de dias: 1,126


,base,registros_antes_da_janela,registros_dentro_da_janela,registros_depois_da_janela
0,alarmes,0,741138,1368
1,corretiva_emergencial,306,10200,37
2,corretiva_programada,78,1085,0
3,preventiva,38,530,1


In [9]:
bases_janela = {}
resumo_filtro_temporal = []

for nome_base, df in bases.items():
    mascara_periodo = df["data_evento"].between(DATA_INICIO_COMUM, DATA_FIM_COMUM)
    bases_janela[nome_base] = df.loc[mascara_periodo].copy()

    resumo_filtro_temporal.append({
        "base": nome_base,
        "linhas_antes": len(df),
        "linhas_depois": len(bases_janela[nome_base]),
        "linhas_removidas": int((~mascara_periodo).sum()),
        "percentual_preservado": round(mascara_periodo.mean() * 100, 2),
    })

display(pd.DataFrame(resumo_filtro_temporal))

,base,linhas_antes,linhas_depois,linhas_removidas,percentual_preservado
0,alarmes,742506,741138,1368,99.82
1,corretiva_emergencial,10543,10200,343,96.75
2,corretiva_programada,1163,1085,78,93.29
3,preventiva,569,530,39,93.15


In [10]:
combinacoes_psd = (
    bases_janela["alarmes"][["estacao_padrao", "plataforma_padrao"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["estacao_padrao", "plataforma_padrao"])
    .reset_index(drop=True)
)

calendario = pd.DataFrame({
    "data_referencia": pd.date_range(DATA_INICIO_COMUM, DATA_FIM_COMUM, freq="D")
})

tabela_mestra = combinacoes_psd.merge(calendario, how="cross")
tabela_mestra["id_psd"] = (
    tabela_mestra["estacao_padrao"]
    + "_P"
    + tabela_mestra["plataforma_padrao"].astype(str)
)
tabela_mestra = tabela_mestra[["data_referencia", "id_psd", "estacao_padrao", "plataforma_padrao"]]
tabela_mestra = tabela_mestra.sort_values(["data_referencia", "id_psd"]).reset_index(drop=True)

print(f"Combinações PSD: {len(combinacoes_psd)}")
print(f"Dias no calendário: {len(calendario):,}")
print(f"Linhas da tabela mestra: {len(tabela_mestra):,}")
display(combinacoes_psd)
display(tabela_mestra.head())

Combinações PSD: 22
Dias no calendário: 1,126
Linhas da tabela mestra: 24,772


,estacao_padrao,plataforma_padrao
0,BUT,1
1,BUT,2
2,FAL,1
3,FAL,2
4,FRA,1
5,FRA,2
6,FRE,1
7,FRE,2
8,HIG,1
9,HIG,2


,data_referencia,id_psd,estacao_padrao,plataforma_padrao
0,2023-02-27,BUT_P1,BUT,1
1,2023-02-27,BUT_P2,BUT,2
2,2023-02-27,FAL_P1,FAL,1
3,2023-02-27,FAL_P2,FAL,2
4,2023-02-27,FRA_P1,FRA,1


In [11]:
alarmes_eventos = bases_janela["alarmes"].copy()

alarmes_eventos["anomalia_critica_30s"] = alarmes_eventos["duracao_calculada_seg"] > 30
hora_alarme = alarmes_eventos["data_hora_evento"].dt.hour
alarmes_eventos["critica_pico_manha"] = (
    alarmes_eventos["anomalia_critica_30s"] & hora_alarme.between(6, 9)
)
alarmes_eventos["critica_pico_tarde"] = (
    alarmes_eventos["anomalia_critica_30s"] & hora_alarme.between(16, 19)
)

alarmes_diarios = (
    alarmes_eventos
    .groupby(["data_evento", "estacao_padrao", "plataforma_padrao"], as_index=False)
    .agg(
        qtd_alarmes_total=("tag", "size"),
        qtd_anomalias_criticas_30s=("anomalia_critica_30s", "sum"),
        qtd_criticas_pico_manha=("critica_pico_manha", "sum"),
        qtd_criticas_pico_tarde=("critica_pico_tarde", "sum"),
        duracao_total_alarmes_seg=("duracao_calculada_seg", "sum"),
        duracao_media_alarmes_seg=("duracao_calculada_seg", "mean"),
        duracao_maxima_alarme_seg=("duracao_calculada_seg", "max"),
        qtd_portas_com_alarme=("porta_padrao", "nunique"),
    )
    .rename(columns={"data_evento": "data_referencia"})
)

print(f"Linhas de eventos de alarme: {len(alarmes_eventos):,}")
print(f"Linhas do resumo diário: {len(alarmes_diarios):,}")
display(alarmes_diarios.head())

Linhas de eventos de alarme: 741,138
Linhas do resumo diário: 11,592


,data_referencia,estacao_padrao,plataforma_padrao,qtd_alarmes_total,qtd_anomalias_criticas_30s,qtd_criticas_pico_manha,qtd_criticas_pico_tarde,duracao_total_alarmes_seg,duracao_media_alarmes_seg,duracao_maxima_alarme_seg,qtd_portas_com_alarme
0,2023-02-27,FRE,1,27,21,21,0,"250,096.87","9,262.85","15,159.45",3
1,2023-02-27,FRE,2,4,0,0,0,3.97,0.99,1.00,1
2,2023-02-28,FRE,1,6,0,0,0,5.97,0.99,1.00,3
3,2023-02-28,FRE,2,13,0,0,0,23.03,1.77,11.00,6
4,2023-03-01,FRE,1,9,3,0,0,585.98,65.11,318.01,2


In [12]:
chaves_tabela_mestra = ["data_referencia", "estacao_padrao", "plataforma_padrao"]
linhas_antes_juncao = len(tabela_mestra)

tabela_mestra = tabela_mestra.merge(
    alarmes_diarios,
    on=chaves_tabela_mestra,
    how="left",
    validate="one_to_one",
)

colunas_contagem_alarmes = [
    "qtd_alarmes_total",
    "qtd_anomalias_criticas_30s",
    "qtd_criticas_pico_manha",
    "qtd_criticas_pico_tarde",
    "qtd_portas_com_alarme",
]
colunas_duracao_alarmes = [
    "duracao_total_alarmes_seg",
    "duracao_media_alarmes_seg",
    "duracao_maxima_alarme_seg",
]

tabela_mestra[colunas_contagem_alarmes] = (
    tabela_mestra[colunas_contagem_alarmes].fillna(0).astype("Int32")
)
tabela_mestra[colunas_duracao_alarmes] = tabela_mestra[colunas_duracao_alarmes].fillna(0.0)

assert len(tabela_mestra) == linhas_antes_juncao
assert not tabela_mestra.duplicated(chaves_tabela_mestra).any()

print(f"Linhas preservadas após a junção: {len(tabela_mestra):,}")
print(f"Dias com pelo menos um alarme: {(tabela_mestra['qtd_alarmes_total'] > 0).sum():,}")
display(tabela_mestra.head())

Linhas preservadas após a junção: 24,772
Dias com pelo menos um alarme: 11,592


,data_referencia,id_psd,estacao_padrao,plataforma_padrao,qtd_alarmes_total,qtd_anomalias_criticas_30s,qtd_criticas_pico_manha,qtd_criticas_pico_tarde,duracao_total_alarmes_seg,duracao_media_alarmes_seg,duracao_maxima_alarme_seg,qtd_portas_com_alarme
0,2023-02-27,BUT_P1,BUT,1,0,0,0,0,0.00,0.00,0.00,0
1,2023-02-27,BUT_P2,BUT,2,0,0,0,0,0.00,0.00,0.00,0
2,2023-02-27,FAL_P1,FAL,1,0,0,0,0,0.00,0.00,0.00,0
3,2023-02-27,FAL_P2,FAL,2,0,0,0,0,0.00,0.00,0.00,0
4,2023-02-27,FRA_P1,FRA,1,0,0,0,0,0.00,0.00,0.00,0


In [13]:
corretivas_emergenciais_eventos = bases_janela["corretiva_emergencial"].copy()

mascara_chaves_validas = (
    corretivas_emergenciais_eventos["estacao_padrao"].notna()
    & corretivas_emergenciais_eventos["plataforma_padrao"].notna()
)
corretivas_emergenciais_sem_chave = corretivas_emergenciais_eventos.loc[~mascara_chaves_validas].copy()
corretivas_emergenciais_com_chave = corretivas_emergenciais_eventos.loc[mascara_chaves_validas].copy()

corretivas_emergenciais_diarias = (
    corretivas_emergenciais_com_chave
    .groupby(["data_evento", "estacao_padrao", "plataforma_padrao"], as_index=False)
    .agg(
        qtd_corretivas_emergenciais=("nota", "size"),
        qtd_falhas_confirmadas=("falha_flag", "sum"),
        qtd_indisponibilidades=("indisponivel_flag", "sum"),
        duracao_total_corretivas_horas=("duracao_avaria_horas", "sum"),
        duracao_maxima_corretiva_horas=("duracao_avaria_horas", "max"),
    )
    .rename(columns={"data_evento": "data_referencia"})
)

print(f"Ordens emergenciais na janela: {len(corretivas_emergenciais_eventos):,}")
print(f"Ordens agregadas com estação e plataforma: {len(corretivas_emergenciais_com_chave):,}")
print(f"Ordens preservadas para auditoria, mas sem chave completa: {len(corretivas_emergenciais_sem_chave):,}")
print(f"Falhas sem chave completa: {int(corretivas_emergenciais_sem_chave['falha_flag'].sum()):,}")
display(corretivas_emergenciais_diarias.head())

Ordens emergenciais na janela: 10,200
Ordens agregadas com estação e plataforma: 9,991
Ordens preservadas para auditoria, mas sem chave completa: 209
Falhas sem chave completa: 51


,data_referencia,estacao_padrao,plataforma_padrao,qtd_corretivas_emergenciais,qtd_falhas_confirmadas,qtd_indisponibilidades,duracao_total_corretivas_horas,duracao_maxima_corretiva_horas
0,2023-02-27,FAL,1,1,1,1,0.98,0.98
1,2023-02-27,PTA,1,1,1,1,0.33,0.33
2,2023-02-27,REP,1,1,1,1,1.25,1.25
3,2023-02-27,VSO,2,1,1,1,0.92,0.92
4,2023-02-28,FAL,1,1,1,0,0.92,0.92


In [14]:
linhas_antes_juncao = len(tabela_mestra)
total_falhas_antes_juncao = int(corretivas_emergenciais_diarias["qtd_falhas_confirmadas"].sum())

tabela_mestra = tabela_mestra.merge(
    corretivas_emergenciais_diarias,
    on=chaves_tabela_mestra,
    how="left",
    validate="one_to_one",
)

colunas_contagem_corretivas = [
    "qtd_corretivas_emergenciais",
    "qtd_falhas_confirmadas",
    "qtd_indisponibilidades",
]
colunas_duracao_corretivas = [
    "duracao_total_corretivas_horas",
    "duracao_maxima_corretiva_horas",
]

tabela_mestra[colunas_contagem_corretivas] = (
    tabela_mestra[colunas_contagem_corretivas].fillna(0).astype("Int16")
)
tabela_mestra[colunas_duracao_corretivas] = tabela_mestra[colunas_duracao_corretivas].fillna(0.0)

assert len(tabela_mestra) == linhas_antes_juncao
assert not tabela_mestra.duplicated(chaves_tabela_mestra).any()
assert int(tabela_mestra["qtd_falhas_confirmadas"].sum()) == total_falhas_antes_juncao

print(f"Linhas preservadas após a junção: {len(tabela_mestra):,}")
print(f"Falhas confirmadas incorporadas: {total_falhas_antes_juncao:,}")
print(f"Dias-PSD com falha: {(tabela_mestra['qtd_falhas_confirmadas'] > 0).sum():,}")
display(tabela_mestra.loc[tabela_mestra["qtd_falhas_confirmadas"] > 0].head())

Linhas preservadas após a junção: 24,772
Falhas confirmadas incorporadas: 3,045
Dias-PSD com falha: 2,593


,data_referencia,id_psd,estacao_padrao,plataforma_padrao,qtd_alarmes_total,qtd_anomalias_criticas_30s,qtd_criticas_pico_manha,qtd_criticas_pico_tarde,duracao_total_alarmes_seg,duracao_media_alarmes_seg,duracao_maxima_alarme_seg,qtd_portas_com_alarme,qtd_corretivas_emergenciais,qtd_falhas_confirmadas,qtd_indisponibilidades,duracao_total_corretivas_horas,duracao_maxima_corretiva_horas
2,2023-02-27,FAL_P1,FAL,1,0,0,0,0,0.00,0.00,0.00,0,1,1,1,0.98,0.98
16,2023-02-27,PTA_P1,PTA,1,0,0,0,0,0.00,0.00,0.00,0,1,1,1,0.33,0.33
18,2023-02-27,REP_P1,REP,1,0,0,0,0,0.00,0.00,0.00,0,1,1,1,1.25,1.25
21,2023-02-27,VSO_P2,VSO,2,0,0,0,0,0.00,0.00,0.00,0,1,1,1,0.92,0.92
24,2023-02-28,FAL_P1,FAL,1,0,0,0,0,0.00,0.00,0.00,0,1,1,0,0.92,0.92


In [15]:
corretivas_programadas_eventos = bases_janela["corretiva_programada"].copy()

mascara_chaves_validas = (
    corretivas_programadas_eventos["estacao_padrao"].notna()
    & corretivas_programadas_eventos["plataforma_padrao"].notna()
)
corretivas_programadas_sem_chave = corretivas_programadas_eventos.loc[~mascara_chaves_validas].copy()
corretivas_programadas_com_chave = corretivas_programadas_eventos.loc[mascara_chaves_validas].copy()

corretivas_programadas_diarias = (
    corretivas_programadas_com_chave
    .groupby(["data_evento", "estacao_padrao", "plataforma_padrao"], as_index=False)
    .agg(
        qtd_corretivas_programadas=("nota", "size"),
        qtd_indisponibilidades_programadas=("indisponivel_flag", "sum"),
        duracao_total_programadas_horas=("duracao_avaria_horas", "sum"),
        duracao_maxima_programada_horas=("duracao_avaria_horas", "max"),
    )
    .rename(columns={"data_evento": "data_referencia"})
)

print(f"Ordens programadas na janela: {len(corretivas_programadas_eventos):,}")
print(f"Ordens agregadas com estação e plataforma: {len(corretivas_programadas_com_chave):,}")
print(f"Ordens preservadas para auditoria, mas sem chave completa: {len(corretivas_programadas_sem_chave):,}")
display(corretivas_programadas_diarias.head())

Ordens programadas na janela: 1,085
Ordens agregadas com estação e plataforma: 1,029
Ordens preservadas para auditoria, mas sem chave completa: 56


,data_referencia,estacao_padrao,plataforma_padrao,qtd_corretivas_programadas,qtd_indisponibilidades_programadas,duracao_total_programadas_horas,duracao_maxima_programada_horas
0,2023-02-27,FRA,2,1,0,407.33,407.33
1,2023-03-01,MBI,1,1,0,1.00,1.00
2,2023-03-07,FAL,1,8,0,"4,521.00","1,584.00"
3,2023-03-07,FAL,2,1,0,"1,584.50","1,584.50"
4,2023-03-07,FRA,2,1,0,9.25,9.25


In [16]:
linhas_antes_juncao = len(tabela_mestra)
total_programadas_antes_juncao = int(corretivas_programadas_diarias["qtd_corretivas_programadas"].sum())

tabela_mestra = tabela_mestra.merge(
    corretivas_programadas_diarias,
    on=chaves_tabela_mestra,
    how="left",
    validate="one_to_one",
)

colunas_contagem_programadas = [
    "qtd_corretivas_programadas",
    "qtd_indisponibilidades_programadas",
]
colunas_duracao_programadas = [
    "duracao_total_programadas_horas",
    "duracao_maxima_programada_horas",
]

tabela_mestra[colunas_contagem_programadas] = (
    tabela_mestra[colunas_contagem_programadas].fillna(0).astype("Int16")
)
tabela_mestra[colunas_duracao_programadas] = tabela_mestra[colunas_duracao_programadas].fillna(0.0)

assert len(tabela_mestra) == linhas_antes_juncao
assert not tabela_mestra.duplicated(chaves_tabela_mestra).any()
assert int(tabela_mestra["qtd_corretivas_programadas"].sum()) == total_programadas_antes_juncao

print(f"Linhas preservadas após a junção: {len(tabela_mestra):,}")
print(f"Corretivas programadas incorporadas: {total_programadas_antes_juncao:,}")
print(f"Dias-PSD com corretiva programada: {(tabela_mestra['qtd_corretivas_programadas'] > 0).sum():,}")
display(tabela_mestra.loc[tabela_mestra["qtd_corretivas_programadas"] > 0].head())

Linhas preservadas após a junção: 24,772
Corretivas programadas incorporadas: 1,029
Dias-PSD com corretiva programada: 700


,data_referencia,id_psd,estacao_padrao,plataforma_padrao,qtd_alarmes_total,qtd_anomalias_criticas_30s,qtd_criticas_pico_manha,qtd_criticas_pico_tarde,duracao_total_alarmes_seg,duracao_media_alarmes_seg,duracao_maxima_alarme_seg,qtd_portas_com_alarme,qtd_corretivas_emergenciais,qtd_falhas_confirmadas,qtd_indisponibilidades,duracao_total_corretivas_horas,duracao_maxima_corretiva_horas,qtd_corretivas_programadas,qtd_indisponibilidades_programadas,duracao_total_programadas_horas,duracao_maxima_programada_horas
5,2023-02-27,FRA_P2,FRA,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,1,0,407.33,407.33
56,2023-03-01,MBI_P1,MBI,1,0,0,0,0,0.00,0.00,0.00,0,1,1,0,0.73,0.73,1,0,1.00,1.00
178,2023-03-07,FAL_P1,FAL,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,8,0,"4,521.00","1,584.00"
179,2023-03-07,FAL_P2,FAL,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,1,0,"1,584.50","1,584.50"
181,2023-03-07,FRA_P2,FRA,2,0,0,0,0,0.00,0.00,0.00,0,3,2,0,2.40,1.40,1,0,9.25,9.25


In [17]:
preventivas_eventos = bases_janela["preventiva"].copy()
preventivas_eventos["nao_cancelada_flag"] = ~preventivas_eventos["cancelada"]

mascara_chaves_validas = (
    preventivas_eventos["estacao_padrao"].notna()
    & preventivas_eventos["plataforma_padrao"].notna()
)
preventivas_sem_chave = preventivas_eventos.loc[~mascara_chaves_validas].copy()
preventivas_com_chave = preventivas_eventos.loc[mascara_chaves_validas].copy()

preventivas_diarias = (
    preventivas_com_chave
    .groupby(["data_evento", "estacao_padrao", "plataforma_padrao"], as_index=False)
    .agg(
        qtd_preventivas=("ordem", "size"),
        qtd_preventivas_canceladas=("cancelada", "sum"),
        qtd_preventivas_nao_canceladas=("nao_cancelada_flag", "sum"),
        duracao_total_preventivas_horas=("duracao_h", "sum"),
        duracao_maxima_preventiva_horas=("duracao_h", "max"),
    )
    .rename(columns={"data_evento": "data_referencia"})
)

print(f"Ordens preventivas na janela: {len(preventivas_eventos):,}")
print(f"Ordens agregadas com estação e plataforma: {len(preventivas_com_chave):,}")
print(f"Ordens preservadas para auditoria, mas sem chave completa: {len(preventivas_sem_chave):,}")
print(f"Preventivas canceladas com chave completa: {int(preventivas_com_chave['cancelada'].sum()):,}")
display(preventivas_diarias.head())

Ordens preventivas na janela: 530
Ordens agregadas com estação e plataforma: 292
Ordens preservadas para auditoria, mas sem chave completa: 238
Preventivas canceladas com chave completa: 0


,data_referencia,estacao_padrao,plataforma_padrao,qtd_preventivas,qtd_preventivas_canceladas,qtd_preventivas_nao_canceladas,duracao_total_preventivas_horas,duracao_maxima_preventiva_horas
0,2023-03-06,FAL,1,1,0,1,76.52,76.52
1,2023-03-06,FAL,2,1,0,1,76.52,76.52
2,2023-03-23,MBI,1,1,0,1,76.52,76.52
3,2023-03-23,MBI,2,1,0,1,76.52,76.52
4,2023-03-27,PTA,1,1,0,1,172.52,172.52


In [18]:
linhas_antes_juncao = len(tabela_mestra)
total_preventivas_antes_juncao = int(preventivas_diarias["qtd_preventivas"].sum())

tabela_mestra = tabela_mestra.merge(
    preventivas_diarias,
    on=chaves_tabela_mestra,
    how="left",
    validate="one_to_one",
)

colunas_contagem_preventivas = [
    "qtd_preventivas",
    "qtd_preventivas_canceladas",
    "qtd_preventivas_nao_canceladas",
]
colunas_duracao_preventivas = [
    "duracao_total_preventivas_horas",
    "duracao_maxima_preventiva_horas",
]

tabela_mestra[colunas_contagem_preventivas] = (
    tabela_mestra[colunas_contagem_preventivas].fillna(0).astype("Int16")
)
tabela_mestra[colunas_duracao_preventivas] = tabela_mestra[colunas_duracao_preventivas].fillna(0.0)

assert len(tabela_mestra) == linhas_antes_juncao
assert not tabela_mestra.duplicated(chaves_tabela_mestra).any()
assert int(tabela_mestra["qtd_preventivas"].sum()) == total_preventivas_antes_juncao

print(f"Linhas preservadas após a junção: {len(tabela_mestra):,}")
print(f"Preventivas incorporadas: {total_preventivas_antes_juncao:,}")
print(f"Dias-PSD com preventiva: {(tabela_mestra['qtd_preventivas'] > 0).sum():,}")
display(tabela_mestra.loc[tabela_mestra["qtd_preventivas"] > 0].head())

Linhas preservadas após a junção: 24,772
Preventivas incorporadas: 292
Dias-PSD com preventiva: 292


,data_referencia,id_psd,estacao_padrao,plataforma_padrao,qtd_alarmes_total,qtd_anomalias_criticas_30s,qtd_criticas_pico_manha,qtd_criticas_pico_tarde,duracao_total_alarmes_seg,duracao_media_alarmes_seg,duracao_maxima_alarme_seg,qtd_portas_com_alarme,qtd_corretivas_emergenciais,qtd_falhas_confirmadas,qtd_indisponibilidades,duracao_total_corretivas_horas,duracao_maxima_corretiva_horas,qtd_corretivas_programadas,qtd_indisponibilidades_programadas,duracao_total_programadas_horas,duracao_maxima_programada_horas,qtd_preventivas,qtd_preventivas_canceladas,qtd_preventivas_nao_canceladas,duracao_total_preventivas_horas,duracao_maxima_preventiva_horas
156,2023-03-06,FAL_P1,FAL,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,1,0,1,76.52,76.52
157,2023-03-06,FAL_P2,FAL,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,1,0,1,76.52,76.52
540,2023-03-23,MBI_P1,MBI,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,1,0,1,76.52,76.52
541,2023-03-23,MBI_P2,MBI,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,1,0,1,76.52,76.52
632,2023-03-27,PTA_P1,PTA,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,1,0,1,172.52,172.52


In [19]:
linhas_esperadas = len(combinacoes_psd) * len(calendario)
dias_por_psd = tabela_mestra.groupby("id_psd")["data_referencia"].nunique()
colunas_metricas = tabela_mestra.select_dtypes(include="number").columns.drop("plataforma_padrao")

assert len(tabela_mestra) == linhas_esperadas
assert tabela_mestra["id_psd"].nunique() == len(combinacoes_psd)
assert tabela_mestra["data_referencia"].nunique() == len(calendario)
assert dias_por_psd.nunique() == 1 and dias_por_psd.iloc[0] == len(calendario)
assert not tabela_mestra.duplicated(chaves_tabela_mestra).any()
assert not tabela_mestra.isna().any().any()
assert (tabela_mestra[colunas_metricas] >= 0).all().all()
assert (tabela_mestra["qtd_falhas_confirmadas"] <= tabela_mestra["qtd_corretivas_emergenciais"]).all()

auditoria_final = pd.DataFrame({
    "verificacao": [
        "linhas",
        "colunas",
        "sistemas_psd",
        "dias",
        "chaves_duplicadas",
        "valores_nulos",
        "data_minima",
        "data_maxima",
    ],
    "resultado": [
        len(tabela_mestra),
        tabela_mestra.shape[1],
        tabela_mestra["id_psd"].nunique(),
        tabela_mestra["data_referencia"].nunique(),
        int(tabela_mestra.duplicated(chaves_tabela_mestra).sum()),
        int(tabela_mestra.isna().sum().sum()),
        tabela_mestra["data_referencia"].min(),
        tabela_mestra["data_referencia"].max(),
    ],
})

display(auditoria_final)
display(tabela_mestra.dtypes.rename("tipo").to_frame())

,verificacao,resultado
0,linhas,24772
1,colunas,26
2,sistemas_psd,22
3,dias,1126
4,chaves_duplicadas,0
5,valores_nulos,0
6,data_minima,2023-02-27 00:00:00
7,data_maxima,2026-03-28 00:00:00


,tipo
data_referencia,datetime64[us]
id_psd,string
estacao_padrao,string
plataforma_padrao,Int8
qtd_alarmes_total,Int32
qtd_anomalias_criticas_30s,Int32
qtd_criticas_pico_manha,Int32
qtd_criticas_pico_tarde,Int32
duracao_total_alarmes_seg,float64
duracao_media_alarmes_seg,float64


In [20]:
CAMINHO_TABELA_MESTRA = Path("../../data/processados/modelo-07/tabela_mestra_psd.csv")
CAMINHO_TABELA_MESTRA.parent.mkdir(parents=True, exist_ok=True)

tabela_mestra.to_csv(
    CAMINHO_TABELA_MESTRA,
    index=False,
    date_format="%Y-%m-%d",
)

print(f"Tabela mestra salva em: {CAMINHO_TABELA_MESTRA.resolve()}")
print(f"Linhas exportadas: {len(tabela_mestra):,}")
print(f"Tamanho do arquivo: {CAMINHO_TABELA_MESTRA.stat().st_size / 1024**2:.2f} MB")

Tabela mestra salva em: /home/cardoso/GitHub/g04/data/processados/modelo-07/tabela_mestra_psd.csv
Linhas exportadas: 24,772
Tamanho do arquivo: 2.46 MB
